# 04 · Live Paper Trades vs Testing (Backtest) — Divergence Analysis

Compares the **live paper trading** results (`live_paper_trades.csv`, real 2026 runs on EC2) against the
**backtest** results (`paper_trades_full_history.csv`, 2024–2026) that the system was validated on.

**Question:** the backtest is strongly profitable — why is live losing money, and where exactly does it diverge?

Sections:
1. Load both datasets (handles the live CSV's mixed 22/23/24-column schema)
2. Headline summary — live vs testing
3. Year-by-year (2024 / 2025 / 2026)
4. Direction split — longs vs shorts (where the leak is)
5. Driver strategy: CAMARILLA collapse
6. Entry lag & price drift (live-only mechanics)
7. Exit-reason / intra-candle optimism
8. Time-of-day
9. Same-day worked example (DIXON)
10. Cumulative P&L
11. Findings

In [1]:
import sys, csv
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from config.settings import TRADE_LOG_DIR

LIVE_FILE = TRADE_LOG_DIR / "live_paper_trades.csv"
TEST_FILE = TRADE_LOG_DIR / "paper_trades_full_history.csv"   # full 2024-2026 backtest record

# ── Robust live loader: live_paper_trades.csv mixes 3 schemas ──────────────────
# 22 cols (oldest): no entry_time, no strategy_entry
# 23 cols         : + entry_time
# 24 cols (current): + entry_time + strategy_entry
SCHEMA_24 = ["date","symbol","signal_time","entry_time","strategy_entry","entry_price",
    "quantity","position_rs","stop_loss","target","rr","strategies_fired","agreeing_count",
    "composite_score","driver_strategy","reason","exit_time","exit_price","exit_reason",
    "result","pnl_rs","pnl_pct","predicted_win_pct","conviction_tier"]

def load_live(path):
    rows = []
    with open(path, newline="") as f:
        raw = list(csv.reader(f))
    for row in raw[1:]:
        n = len(row)
        if n == 24:
            d = dict(zip(SCHEMA_24, row))
        elif n == 23:
            d = dict(zip(SCHEMA_24[:4] + SCHEMA_24[5:], row)); d["strategy_entry"] = ""
        elif n == 22:
            d = dict(zip(["date","symbol","signal_time"] + SCHEMA_24[5:], row))
            d["entry_time"] = ""; d["strategy_entry"] = ""
        else:
            continue
        rows.append(d)
    df = pd.DataFrame(rows)
    for col in ["entry_price","strategy_entry","quantity","position_rs","stop_loss","target",
                "rr","agreeing_count","composite_score","exit_price","pnl_rs","pnl_pct","predicted_win_pct"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["date"] = pd.to_datetime(df["date"])
    return df

def tmin(s):
    try:
        h, m = map(int, str(s).split(":")); return h*60 + m
    except Exception:
        return np.nan

def enrich(df, is_live):
    df = df.copy()
    df["year"] = df["date"].dt.year
    df["sig_min"]  = df["signal_time"].apply(tmin)
    df["exit_min"] = df["exit_time"].apply(tmin)
    if is_live:
        df["entry_min"] = df["entry_time"].apply(tmin)
        df["entry_lag"] = df["entry_min"] - df["sig_min"]
        df["drift_pct"] = np.where(df["strategy_entry"] > 0,
            (df["entry_price"] - df["strategy_entry"]) / df["strategy_entry"] * 100, np.nan)
        # infer direction (no direction column in live schema): LONG if target>entry
        df["direction"] = np.where(df["target"] > df["entry_price"], "LONG",
                          np.where(df["target"] < df["entry_price"], "SHORT", "?"))
    else:
        df["entry_min"] = df["sig_min"]; df["entry_lag"] = 0.0; df["drift_pct"] = 0.0
    df["hold_min"] = df["exit_min"] - df["entry_min"]
    def slot(m):
        if pd.isna(m): return "unknown"
        h, mm = int(m)//60, int(m)%60
        base = mm < 30
        eh, em = (h,30) if base else (h+1,0)
        return f"{h:02d}:{'00' if base else '30'}-{eh:02d}:{em:02d}"
    df["time_slot"] = df["sig_min"].apply(slot)
    return df

live = enrich(load_live(LIVE_FILE), is_live=True)
test = enrich(pd.read_csv(TEST_FILE, parse_dates=["date"]), is_live=False)

print(f"LIVE : {len(live):4} trades  {live['date'].min().date()} -> {live['date'].max().date()}")
print(f"TEST : {len(test):4} trades  {test['date'].min().date()} -> {test['date'].max().date()}")
print("LIVE by year:", dict(live.groupby('year').size()))
print("TEST by year:", dict(test.groupby('year').size()))

LIVE :   37 trades  2026-06-15 -> 2026-07-14
TEST : 2905 trades  2022-01-03 -> 2026-06-29
LIVE by year: {2026: np.int64(37)}
TEST by year: {2022: np.int64(494), 2023: np.int64(484), 2024: np.int64(493), 2025: np.int64(975), 2026: np.int64(459)}


## 2. Headline summary — live vs testing

In [2]:
def summary(df):
    n = len(df)
    if n == 0: return {}
    return {
        "trades":   n,
        "exact_%":  round((df.result=="EXACT_WIN").mean()*100, 1),
        "win_%":    round((df.result=="WIN").mean()*100, 1),
        "loss_%":   round((df.result=="LOSS").mean()*100, 1),
        "eff_WR":   round((df.result!="LOSS").mean()*100, 1),
        "stop_%":   round((df.exit_reason=="STOP_HIT").mean()*100, 1),
        "total_pnl":int(df.pnl_rs.sum()),
        "avg_pnl":  int(df.pnl_rs.mean()),
    }

rows = {
    "LIVE 2026 (real)":      summary(live),
    "TEST 2024":             summary(test[test.year==2024]),
    "TEST 2025":             summary(test[test.year==2025]),
    "TEST 2026":             summary(test[test.year==2026]),
    "TEST 2024-26 combined": summary(test[test.year.isin([2024,2025,2026])]),
}
comp = pd.DataFrame(rows).T
comp

,trades,exact_%,win_%,loss_%,eff_WR,stop_%,total_pnl,avg_pnl
LIVE 2026 (real),37.0,24.3,18.9,56.8,43.2,43.2,-28890.0,-780.0
TEST 2024,493.0,47.3,19.7,33.1,66.9,24.5,1454192.0,2949.0
TEST 2025,975.0,47.9,17.8,34.3,65.7,25.4,2296573.0,2355.0
TEST 2026,459.0,40.7,22.0,37.3,62.7,29.4,1076910.0,2346.0
TEST 2024-26 combined,1927.0,46.0,19.3,34.7,65.3,26.2,4827677.0,2505.0


**Read:** live turns a backtested ~65% win rate into 43%, and the stop-out rate nearly doubles.
Exact-target hits are halved — the trades meant to reach target are hitting stop instead.

## 3. Year-by-year bar comparison

In [3]:
labels = ["TEST 2024","TEST 2025","TEST 2026","LIVE 2026"]
subsets = [test[test.year==2024], test[test.year==2025], test[test.year==2026], live]
eff = [(s.result!="LOSS").mean()*100 for s in subsets]
avg = [s.pnl_rs.mean() for s in subsets]
colors = ["#3fd07f","#3fd07f","#3fd07f","#ff6b6b"]

fig = make_subplots(rows=1, cols=2, subplot_titles=["Effective win rate (%)","Avg P&L per trade (Rs)"])
fig.add_trace(go.Bar(x=labels, y=eff, marker_color=colors,
    text=[f"{v:.1f}%" for v in eff], textposition="auto"), row=1, col=1)
fig.add_trace(go.Bar(x=labels, y=avg, marker_color=colors,
    text=[f"{v:,.0f}" for v in avg], textposition="auto"), row=1, col=2)
fig.add_hline(y=50, row=1, col=1, line_dash="dash", line_color="gray")
fig.add_hline(y=0,  row=1, col=2, line_dash="dash", line_color="gray")
fig.update_layout(template="plotly_dark", height=420, showlegend=False,
    title="Backtest (green) vs Live (red) — 2024 to 2026")
fig.show()

## 4. Direction split — where the money leaks (longs vs shorts)

In [4]:
def dir_summary(df, label):
    out = {}
    for dr in ["LONG","SHORT"]:
        s = df[df.direction == dr]
        if len(s):
            out[f"{label} {dr}"] = summary(s)
    return out

d = {}
d.update(dir_summary(test[test.year==2026], "TEST26"))
d.update(dir_summary(live, "LIVE"))
dsplit = pd.DataFrame(d).T[["trades","exact_%","eff_WR","total_pnl","avg_pnl"]]
dsplit

,trades,exact_%,eff_WR,total_pnl,avg_pnl
TEST26 LONG,224.0,24.6,47.8,324876.0,1450.0
TEST26 SHORT,235.0,56.2,77.0,752034.0,3200.0
LIVE LONG,21.0,28.6,33.3,7861.0,374.0
LIVE SHORT,16.0,18.8,56.2,-36752.0,-2297.0


**The core inversion:** in the backtest, **shorts are the edge** (77% WR, 56% exact, +Rs 3,200/trade).
Live shorts hit exact target only 18.8% of the time and **lose Rs 2,297/trade** — a 56% "win rate" that still
bleeds because the wins are tiny and the losses are full-stop. Live longs roughly match backtest longs; the
short book is where live falls apart.

## 5. Driver strategy — CAMARILLA collapse (51% of live trades)

In [5]:
def strat_table(df):
    g = df.groupby("driver_strategy").agg(
        n=("result","count"),
        exact=("result", lambda x:(x=="EXACT_WIN").mean()*100),
        eff=("result", lambda x:(x!="LOSS").mean()*100),
        avg_pnl=("pnl_rs","mean")).round(1)
    return g

lw = strat_table(live)
tw = strat_table(test[test.year==2026])
cmp = lw.join(tw, lsuffix="_LIVE", rsuffix="_TEST", how="left").fillna(0)
cmp = cmp[["n_LIVE","eff_LIVE","exact_LIVE","avg_pnl_LIVE","eff_TEST","exact_TEST","avg_pnl_TEST"]]
cmp.sort_values("n_LIVE", ascending=False)

,n_LIVE,eff_LIVE,exact_LIVE,avg_pnl_LIVE,eff_TEST,exact_TEST,avg_pnl_TEST
driver_strategy,,,,,,,
CAMARILLA,19,42.1,21.1,74.8,74.5,64.6,4056.2
RSI-EXT,4,0.0,0.0,-9256.3,0.0,0.0,0.0
VPOC,3,66.7,66.7,5539.0,51.5,34.8,2307.2
FIRST-CANDLE,3,33.3,0.0,-2623.1,50.9,21.4,384.7
CPR,3,100.0,100.0,1026.5,100.0,100.0,-649.0
BOLLINGER,2,50.0,0.0,-1418.2,100.0,0.0,4024.0
ASC-TRI,1,0.0,0.0,-479.9,30.0,0.0,-1139.1
INTRADAY-STRUCT,1,0.0,0.0,-3224.8,0.0,0.0,0.0
SUPERTREND,1,100.0,0.0,1427.8,100.0,0.0,1200.8


CAMARILLA drives **19 of 37** live trades (the most-selected driver) and is exactly the strategy whose
backtested **84% win rate collapses to 42% live** (70% → 21% exact). The system concentrates capital in the
strategy most damaged by the live/backtest gap. RSI-EXT (4 trades, −Rs 9,256 avg) and INTRADAY-STRUCT also
appear live but barely exist in the 2026 backtest.

## 6. Entry lag & price drift — live-only mechanics

In [6]:
print("Signal -> actual fill lag (live):")
print(f"   median {live.entry_lag.median():.0f} min | mean {live.entry_lag.mean():.1f} min | max {live.entry_lag.max():.0f} min")
print()
print("Entry price drift vs strategy's theoretical entry (live):")
print(f"   median {live.drift_pct.median():+.2f}% | mean {live.drift_pct.mean():+.2f}% | |mean| {live.drift_pct.abs().mean():.2f}%")
print()

live["lag_bkt"] = pd.cut(live.entry_lag, [-1,4,10,30,999], labels=["0-4m","5-10m","11-30m",">30m"])
lag_wr = live.groupby("lag_bkt", observed=True).agg(
    n=("result","count"),
    eff_WR=("result", lambda x:(x!="LOSS").mean()*100),
    avg_pnl=("pnl_rs","mean")).round(1)
print("Effective WR by entry-lag bucket (live):")
print(lag_wr.to_string())

Signal -> actual fill lag (live):
   median 7 min | mean 16.2 min | max 146 min

Entry price drift vs strategy's theoretical entry (live):
   median +0.33% | mean +0.81% | |mean| 0.99%

Effective WR by entry-lag bucket (live):
          n  eff_WR  avg_pnl
lag_bkt                     
5-10m    26    34.6  -1668.2
11-30m    4    50.0    -34.6
>30m      5    60.0   -114.0


The backtester assumes an **instant fill at `signal_time`** and starts checking exits on the next candle.
Live only places the trade at bar-close after the scan finishes — a median 7 min (up to 146 min) later — and
then **re-anchors the entry to the live price** (`live_engine.py:256`), a median +1.3% away from the strategy's
own level. Targets sit ~1–1.5% away, so this drift can put you in almost *at* the target.

## 7. Exit reasons — intra-candle optimism

In [7]:
ex_live = live.exit_reason.value_counts(normalize=True).mul(100).round(1)
ex_test = test[test.year==2026].exit_reason.value_counts(normalize=True).mul(100).round(1)
ex = pd.DataFrame({"LIVE_%":ex_live, "TEST2026_%":ex_test}).fillna(0)
print(ex.to_string())
print()
print("Backtest _simulate_outcome (engine.py:466-471) checks high>=target BEFORE low<=stop within the")
print("same 5-min bar -> whipsaw bars are scored as target hits. Live monitors real ticks and takes")
print("whichever came first (usually the stop). Result: STOP_HIT jumps from ~29% (test) to 43% (live).")

             LIVE_%  TEST2026_%
exit_reason                    
STOP_HIT       43.2        29.4
TARGET_HIT     24.3        40.7
TIME_EXIT      32.4        29.8

Backtest _simulate_outcome (engine.py:466-471) checks high>=target BEFORE low<=stop within the
same 5-min bar -> whipsaw bars are scored as target hits. Live monitors real ticks and takes
whichever came first (usually the stop). Result: STOP_HIT jumps from ~29% (test) to 43% (live).


## 8. Time-of-day — the edge lives on the open

In [8]:
def slot_wr(df):
    g = df.groupby("time_slot").agg(
        n=("result","count"),
        eff_WR=("result", lambda x:(x!="LOSS").mean()*100)).round(1)
    return g

order = ["09:00-09:30","09:15-09:30","09:30-10:00","10:00-10:30","10:30-11:00"]
ts_test = slot_wr(test[test.year==2026])
ts_live = slot_wr(live)
print("BACKTEST 2026 by signal window:"); print(ts_test.sort_values("n", ascending=False).head(6).to_string())
print()
print("LIVE by signal window:");          print(ts_live.sort_values("n", ascending=False).head(6).to_string())
print()
print("Backtest says the 09:15 open is the golden window (~82% WR). Live fires ~92% of its trades there")
print("too but converts far worse -- the open is where entry-lag and re-anchoring hurt most (price moves fastest).")

BACKTEST 2026 by signal window:
               n  eff_WR
time_slot               
09:00-09:30  263    71.5
09:30-10:00  125    49.6
10:00-10:30   27    33.3
10:30-11:00   12    33.3
11:00-11:30   11    72.7
11:30-12:00    7    57.1

LIVE by signal window:
              n  eff_WR
time_slot              
09:00-09:30  34    44.1
09:30-10:00   2    50.0
10:30-11:00   1     0.0

Backtest says the 09:15 open is the golden window (~82% WR). Live fires ~92% of its trades there
too but converts far worse -- the open is where entry-lag and re-anchoring hurt most (price moves fastest).


## 9. Same-day worked example — DIXON, 2026-06-16

In [9]:
cols = ["date","symbol","direction","signal_time","entry_price","stop_loss","target","rr",
        "exit_time","exit_price","exit_reason","pnl_rs"]
print("LIVE DIXON:")
lv = live[live.symbol=="DIXON"].copy()
lv["signal_time"] = lv["signal_time"] + " (fill " + lv["entry_time"] + ")"
print(lv[cols].to_string(index=False))
print()
print("BACKTEST DIXON 2026-06-16:")
bt = test[(test.date=="2026-06-16") & (test.symbol=="DIXON")]
print(bt[cols].to_string(index=False))
print()
print("Same stock, same day, completely different trade: live filled 25 min late at 12,206 (+2.3% above the")
print("strategy's 11,933 entry), which re-anchored stop & target into a much wider, different trade.")

LIVE DIXON:
      date symbol direction        signal_time  entry_price  stop_loss  target   rr exit_time  exit_price exit_reason  pnl_rs
2026-06-16  DIXON      LONG 09:10 (fill 09:35)      12206.0   11445.15 13777.0 2.06     15:15     12206.0   TIME_EXIT -479.89

BACKTEST DIXON 2026-06-16:
      date symbol direction signal_time  entry_price  stop_loss  target  rr exit_time  exit_price exit_reason   pnl_rs
2026-06-16  DIXON      LONG       09:25      12049.0    11852.0 12443.0 2.0     11:45     11852.0    STOP_HIT -8791.35
2026-06-16  DIXON      LONG       09:25      12049.0    11852.0 12443.0 2.0     11:45     11852.0    STOP_HIT -8791.35

Same stock, same day, completely different trade: live filled 25 min late at 12,206 (+2.3% above the
strategy's 11,933 entry), which re-anchored stop & target into a much wider, different trade.


## 10. Cumulative P&L — live vs backtest-2026 (indexed to trade #)

In [10]:
def cum(df):
    s = df.sort_values(["date","signal_time"]).reset_index(drop=True)
    return np.arange(1, len(s)+1), s.pnl_rs.cumsum().values

xt, yt = cum(test[test.year==2026])
xl, yl = cum(live)
fig = go.Figure()
fig.add_trace(go.Scatter(x=xt, y=yt, mode="lines", name="Backtest 2026",
    line=dict(color="#3fd07f", width=2)))
fig.add_trace(go.Scatter(x=xl, y=yl, mode="lines", name="Live 2026",
    line=dict(color="#ff6b6b", width=2)))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(template="plotly_dark", height=440, title="Cumulative P&L by trade number",
    xaxis_title="Trade #", yaxis_title="Cumulative Rs", hovermode="x unified")
fig.show()

## 11. Findings

**Backtest 2024–26:** +Rs 4.83M, 65% win rate, 46% exact-target.
**Live 2026:** −Rs 28,891, 43% win rate, 24% exact-target. The stop-out rate nearly doubles (26% → 43%).

### Five mechanisms behind the gap (ranked by damage)
1. **Re-anchor to live price** — live overwrites `best_sig.entry` with the last price at scan time and shifts
   stop/target (`live_engine.py:256`). Measured drift median +1.3%, up to +2.3%. *You chase price instead of trading the level.*
2. **Entry lag** — backtest fills instantly at `signal_time`; live fills at bar-close, median 7 min (max 146 min) later.
3. **Intra-candle optimism** — backtest checks `high>=target` before `low<=stop` in the same bar (`engine.py:466`),
   booking whipsaw bars as wins. This alone explains the stop-rate jump.
4. **Different book** — live scans partial intraday bars off a live-built watchlist, so it picks different symbols
   than the backtest on the same day. Live is not a noisy copy of the backtest; it's un-validated trades.
5. **Fill quality** — live walks the real order book (partial fills + slippage); backtest assumes a full fill at a flat 0.05%.

### Actions
- Don't re-anchor: use a limit at the strategy entry; skip if price already drifted past it.
- Make the backtest resolve stop-before-target on ambiguous bars, and enter on the *next* bar's open — then re-run 2024–26.
- Quarantine shorts (live shorts: 18.8% exact, −Rs 2,297/trade) until the backtest short edge reproduces live.
- Fix the `live_paper_trades.csv` schema drift (22/23/24-column rows from a header written once while the schema grew).